# Random Forest untuk Klasifikasi Churn Customer

Maulana Anandyta Narayana

### Import Library

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report, roc_auc_score

### Load Data

In [2]:
df = pd.read_csv("/content/data_ecommerce_customer_churn (1).csv")
df

,Tenure,WarehouseToHome,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,DaySinceLastOrder,CashbackAmount,Churn
0,15.0,29.0,4,Laptop & Accessory,3,Single,2,0,7.0,143.32,0
1,7.0,25.0,4,Mobile,1,Married,2,0,7.0,129.29,0
2,27.0,13.0,3,Laptop & Accessory,1,Married,5,0,7.0,168.54,0
3,20.0,25.0,4,Fashion,3,Divorced,7,0,NaN,230.27,0
4,30.0,15.0,4,Others,4,Single,8,0,8.0,322.17,0
...,...,...,...,...,...,...,...,...,...,...,...
3936,28.0,9.0,5,Fashion,3,Married,8,0,1.0,231.86,0
3937,8.0,7.0,2,Mobile Phone,2,Single,4,0,4.0,157.80,0
3938,30.0,6.0,5,Laptop & Accessory,3,Married,3,1,2.0,156.60,0
3939,6.0,NaN,4,Mobile,3,Married,10,1,0.0,124.37,1


###Membaca Info Data

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3941 entries, 0 to 3940
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Tenure                    3747 non-null   float64
 1   WarehouseToHome           3772 non-null   float64
 2   NumberOfDeviceRegistered  3941 non-null   int64  
 3   PreferedOrderCat          3941 non-null   object 
 4   SatisfactionScore         3941 non-null   int64  
 5   MaritalStatus             3941 non-null   object 
 6   NumberOfAddress           3941 non-null   int64  
 7   Complain                  3941 non-null   int64  
 8   DaySinceLastOrder         3728 non-null   float64
 9   CashbackAmount            3941 non-null   float64
 10  Churn                     3941 non-null   int64  
dtypes: float64(4), int64(5), object(2)
memory usage: 338.8+ KB


In [5]:
df.describe()

,Tenure,WarehouseToHome,NumberOfDeviceRegistered,SatisfactionScore,NumberOfAddress,Complain,DaySinceLastOrder,CashbackAmount,Churn
count,3747.000000,3772.000000,3941.000000,3941.000000,3941.000000,3941.000000,3728.000000,3941.000000,3941.000000
mean,10.081398,15.650583,3.679269,3.088302,4.237757,0.282416,4.531652,176.707419,0.171023
std,8.498864,8.452301,1.013938,1.381832,2.626699,0.450232,3.667648,48.791784,0.376576
min,0.000000,5.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000
25%,2.000000,9.000000,3.000000,2.000000,2.000000,0.000000,2.000000,145.700000,0.000000
50%,9.000000,14.000000,4.000000,3.000000,3.000000,0.000000,3.000000,163.340000,0.000000
75%,16.000000,21.000000,4.000000,4.000000,6.000000,1.000000,7.000000,195.250000,0.000000
max,61.000000,127.000000,6.000000,5.000000,22.000000,1.000000,46.000000,324.990000,1.000000


Dari output di atas, dapat diketahui bahwa ada beberapa variabel yang memiliki nilai non-null kurang dari 3941. Artinya, masih ada beberapa variabel yang memiliki missing value. Selanjutnya, akan dilakukan penanganan terhadap missing value tersebut.

Selain itu, dapat diketahui juga ada beberapa variabel yang memiliki nilai max jauh di atas kuartil ke-3 yang berarti variabel-variabel tersebut memiliki outlier. Karena Random Forest bukanlah model yang sensitif (tahan outlier), maka outlier dapat diabaikan.

### Pilih variabel dependen dan independen

In [6]:
y = df["Churn"]
X = df.drop(columns=["Churn"])

### Pisahkan Kolom

In [7]:
num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns

### Handle Missing Value

Variabel numerik yang kosong diganti dengan median sedangkan variabel kategorik yang kosong diganti dengan modus

In [8]:
num_imputer = SimpleImputer(strategy="median")
X[num_cols] = num_imputer.fit_transform(X[num_cols])

cat_imputer = SimpleImputer(strategy="most_frequent")
X[cat_cols] = cat_imputer.fit_transform(X[cat_cols])

### Encoding Variabel Kategorik

In [9]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")

encoded_cat = encoder.fit_transform(X[cat_cols])

encoded_cat_df = pd.DataFrame(
    encoded_cat,
    columns=encoder.get_feature_names_out(cat_cols)
)

### Gabungkan Data

In [10]:
X_final = pd.concat(
    [X[num_cols].reset_index(drop=True), encoded_cat_df.reset_index(drop=True)],
    axis=1
)

### Split Data

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y, test_size=0.2, random_state=42, stratify=y
)

### Grid Search

Grid search dilakukan untuk mencari kombinasi hyperparameter terbaik untuk meningkatkan akurasi model

In [12]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "class_weight": [None, "balanced"]
}

rf = RandomForestClassifier(random_state=42)

grid = GridSearchCV(
    rf,
    param_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, y_train)

Fitting 5 folds for each of 96 candidates, totalling 480 fits


GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42), n_jobs=-1,
             param_grid={'class_weight': [None, 'balanced'],
                         'max_depth': [None, 10, 20, 30],
                         'min_samples_leaf': [1, 2],
                         'min_samples_split': [2, 5],
                         'n_estimators': [100, 200, 300]},
             scoring='roc_auc', verbose=2)

In [13]:
results = pd.DataFrame(grid.cv_results_)

# kombinasi hyperparameter urut dari yang terbaik
results = results.sort_values(by="mean_test_score", ascending=False)

print(results[[
    "mean_test_score",
    "param_n_estimators",
    "param_max_depth",
    "param_min_samples_split",
    "param_min_samples_leaf",
    "param_class_weight"
]].head(10))

    mean_test_score  param_n_estimators param_max_depth  \
86         0.963350                 300              30   
50         0.963350                 300            None   
74         0.963136                 300              20   
49         0.961543                 200            None   
85         0.961543                 200              30   
73         0.961298                 200              20   
2          0.960996                 300            None   
38         0.960996                 300              30   
26         0.960801                 300              20   
36         0.960625                 100              30   

    param_min_samples_split  param_min_samples_leaf param_class_weight  
86                        2                       1           balanced  
50                        2                       1           balanced  
74                        2                       1           balanced  
49                        2                       1       

Diperoleh kombinasi hyperparameter terbaik :
*   n_estimators = 300
*   max_depth = 30
*   min_samples_split = 2
*   min_samples_leaf = 1
*   param_class_weight = balanced  

Kombinasi ini selanjutnya digunakan untuk training model random forest

### Training Model

In [14]:
rf_final = RandomForestClassifier(
    n_estimators=300,
    max_depth=30,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight="balanced",
    random_state=42
)

rf_final.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=30, n_estimators=300,
                       random_state=42)

### Evaluasi Model

In [15]:
y_pred = rf_final.predict(X_test)
y_prob = rf_final.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("\nROC-AUC Score:")
print(roc_auc_score(y_test, y_prob))

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.99      0.96       654
           1       0.92      0.70      0.80       135

    accuracy                           0.94       789
   macro avg       0.93      0.85      0.88       789
weighted avg       0.94      0.94      0.94       789


ROC-AUC Score:
0.965743572318496


Diperoleh:


*   Skor akurasi: 0.94
*   ROC-AUC Score: 0.965743572318496



Artinya, 94% prediksi model benar. Dalam 96,57% kasus, model berhasil “ranking” customer churn lebih tinggi daripada non-churn. Model sangat bagus dalam membedakan kelas.

###Hasil Klasifikasi

In [16]:
cm = confusion_matrix(y_test, y_pred)
cm

array([[646,   8],
       [ 40,  95]])

Berdasarkan confusion matrix, model berhasil mengklasifikasikan sebagian besar pelanggan dengan benar, dengan jumlah true negative sebesar 646 dan true positive sebesar 95. Namun, terdapat 40 kasus false negative yang menunjukkan bahwa model masih gagal mendeteksi sebagian pelanggan yang churn. Hal ini mengindikasikan bahwa meskipun model memiliki akurasi tinggi, kemampuannya dalam mendeteksi pelanggan yang berisiko churn masih perlu ditingkatkan.

### Perbandingan Nilai Skor  AUC Model Train dan Test

Perbandingan dilakukan dengan tujuan untuk mengetahui apakah model tersebut Overfitting atau tidak

In [17]:
# skor AUC train
y_train_prob = rf_final.predict_proba(X_train)[:, 1]
auc_train = roc_auc_score(y_train, y_train_prob)

# skor AUC test
y_test_prob = rf_final.predict_proba(X_test)[:, 1]
auc_test = roc_auc_score(y_test, y_test_prob)

print("AUC Train:", auc_train)
print("AUC Test :", auc_test)

AUC Train: 1.0
AUC Test : 0.965743572318496


DIperoleh skor
*   AUC Train: 1.0
*   AUC Test : 0.965743572318496

Terdapat sedikit selisih antara skor AUC Train dan AUC Test. Artinya, model sedikit overfitting, tetapi masih wajar sehingga bisa diabaikan
